# NFL Draft Analysis: WR Draft Capital vs. NFL Production
**Goal:** Analyze the relationship between draft position and subsequent NFL performance for wide receivers.

**Data Sources Needed:**
1. Draft data (2017-2025 WRs) - Pick number, team, college
2. Combine/pro-day data - Athletic testing
3. NFL production data (NextGen) - Receiving yards, TDs, Separation, etc. (Players with 45+ Targets Between 2017-2025)
4. College Receiving Stats (Top 300 in receiving yards by year)

**Analysis #1: Hit Rate By Round**

I want a tiered system to determine if a receiver is a hit and to what extent that receiver has hit. Below I have estimated tiers: <br>

Elite:	1,200+ yard season OR 2x 1,000+ yard seasons	Top 12 WR <br>
Hit:	1,000+ yard season OR 2x 700+ yard seasons	Solid starter <br>
Role:	500+ yard season OR 30+ YPG career	Contributes <br>
Bust:	Never hit 500 yards in a season	Didn't work out 

**Database Creation**

Before I do anything else, I need to load all the csv files I have and add them as a table in my SQLite database

In [4]:
import sqlite3
import pandas as pd
import os

# Connect to database (will create if doesn't exist)
conn = sqlite3.connect('nfl_analytics.db')
print("Connected to database")

# List of files to load
files = {
    'draft_picks': '../data/NFLDraftPositionWR2011-2025.csv',
    'nextgen': '../data/NextGenWRStats2018-2025.csv',
    'combine': '../data/NFLCombineStats2017-2025.csv',
    'college': '../data/CollegeReceivingStats2017-2025.csv'
}

# Load each file into SQLite
for table_name, file_path in files.items():
    try:
        df = pd.read_csv(file_path)
        df.to_sql(table_name, conn, if_exists='replace', index=False)
        print(f"Loaded {table_name}: {len(df)} rows")
    except Exception as e:
        print(f"Error loading {file_path}: {e}")

conn.close()
print("\nConnection closed")

Connected to database
Loaded draft_picks: 481 rows
Loaded nextgen: 995 rows
Loaded combine: 550 rows
Loaded college: 2700 rows

Connection closed


This next cell will help me determine how to qualify which receivers are elite, hits, role players, or busts. <br>
I am going to get a distribution of receving yards to see what is considered average, above average, and below average. <br>

Before deciding thresholds that make receivers elite, a hit, a role player, or a bust, I need to decide what those metrics are. To do so, I will run a distrubtion on yards and touchdowns of all receivers in the NextGen data. Because NextGen data has players with over 45 targets, all seasons with less than 45 targets will either be counted as a "lost season" and won't go towards the numbers.


Here are some key methods
1. NextGen Stats - Only 45+ targets

If a WR doesn't get 45 targets in a season, they are most likely either:<br>
* Injured
* Not good enough to get on the field
* A rookie who hasn't earned trust yet

Rookies sometimes take time. A 2024 draftee with 40 targets in Year 1 might be fine. We'll handle this with "Too Soon" flags. <br>

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('nfl_analytics.db')

# See yards distribution
query = """
SELECT 
    CASE 
        WHEN YDS >= 1200 THEN '1200+'
        WHEN YDS >= 800 THEN '800-1199'
        WHEN YDS >= 400 THEN '400-799'
        ELSE 'Under 400'
    END as yards_tier,
    COUNT(*) as seasons,
    COUNT(DISTINCT NAME) as players,
    ROUND(AVG(TD), 1) as avg_tds
FROM nextgen
WHERE POS = 'WR'
GROUP BY yards_tier
ORDER BY MIN(YDS);
"""

yards_dist = pd.read_sql_query(query, conn)
print("WR SEASONS BY YARDS TIER")
print("=" * 60)
print(yards_dist.to_string(index=False))

conn.close()

**Single Season Tiers**
Based on my findings, this is how I will determine certain tiers of receivers:

Elite:	    1,200+ yard season OR 10+ TDs	Only 40 players ever reached this <br>
Hit:	    800+ yard season OR 6+ TDs	93 players, avg 6 TDs - clear starter level <br>
Role:	    400+ yard season OR 3+ TDs	199 players, solid contributor <br>
Bust:	    Never hit 400 yards OR never in NextGen	95 players had a sub-400 season <br>
Too Early:  Players drafted from 2024 or 2025 draft who do not reach any tier <br>

**WHY** <br>
Before I get into deeper analytics, I want to really understand the data I am working with. Therefore, I want to find tiers of receivers based on single season achievements
<br>

**First Step**
<br>
The first step is joining the Draft Picks table with the NextGen table

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('nfl_analytics.db')

# Check the join with correct columns
query_check = """
SELECT 
    d.Name as draft_name,
    d.Year as draft_year,
    d.Round,
    n.NAME as nextgen_name,
    n.YEAR as nextgen_year,
    n.YDS,
    n.TD
FROM draft_picks d
LEFT JOIN nextgen n ON d.Name = n.NAME
WHERE d.Year BETWEEN 2017 AND 2025
    AND d.Name NOT LIKE '%pick%'
    AND d.Name NOT LIKE '%Pick%'
LIMIT 30;
"""

sample = pd.read_sql_query(query_check, conn)
print("🔍 SAMPLE OF JOIN RESULTS:")
print(sample.to_string())

# Count how many have matches
query_count = """
SELECT 
    COUNT(DISTINCT d.Name) as total_drafted,
    COUNT(DISTINCT n.NAME) as have_nextgen_data
FROM draft_picks d
LEFT JOIN nextgen n ON d.Name = n.NAME
WHERE d.Year BETWEEN 2017 AND 2025
    AND d.Name NOT LIKE '%pick%';
"""

counts = pd.read_sql_query(query_count, conn)
print("\n📊 MATCH STATS:")
print(counts)

conn.close()

I limited this query to just 30 players, checking to make sure that the query was pulling everything that we wanted.

**REAL DEAL** <br>

Now we have the real deal to work on. In this next query, I will try to find the results that I am looking for!

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('nfl_analytics.db')

query = """
WITH player_tiers AS (
    -- First, classify each player's BEST season
    SELECT 
        n.NAME,
        MAX(CASE 
            WHEN n.YDS >= 1200 OR n.TD >= 10 THEN 'Elite'
            WHEN n.YDS >= 800 OR n.TD >= 6 THEN 'Hit'
            WHEN n.YDS >= 400 OR n.TD >= 3 THEN 'Role'
            ELSE 'Below Role'
        END) as peak_tier,
        MAX(n.YDS) as peak_yards,
        MAX(n.TD) as peak_tds,
        COUNT(DISTINCT n.YEAR) as seasons_in_nextgen
    FROM nextgen n
    WHERE n.POS = 'WR'
    GROUP BY n.NAME
),
drafted_players AS (
    -- Get all drafted WRs 2017-2025
    SELECT 
        d.Name as Player,
        d.Year as draft_year,
        d.Round,
        d.Pick,
        pt.peak_tier,
        pt.peak_yards,
        pt.peak_tds,
        pt.seasons_in_nextgen,
        -- Determine final tier with "Too Early" logic
        CASE 
            -- Too Early (2024-2025 draftees with no Elite/Hit yet)
            WHEN d.Year >= 2024 AND (pt.peak_tier IS NULL OR pt.peak_tier IN ('Below Role', 'Role')) THEN 'Too Early'
            -- Never appeared in NextGen = Bust
            WHEN pt.peak_tier IS NULL THEN 'Bust'
            -- Appeared but never hit Role = Bust
            WHEN pt.peak_tier = 'Below Role' THEN 'Bust'
            -- Everything else
            ELSE pt.peak_tier
        END as final_tier
    FROM draft_picks d
    LEFT JOIN player_tiers pt ON d.Name = pt.NAME
    WHERE d.Year BETWEEN 2017 AND 2025
        AND d.Name NOT LIKE '%pick%'
        AND d.Name NOT LIKE '%Pick%'
        AND d.Name NOT LIKE '%selection%'
        AND d.Name NOT LIKE '%Forfeit%'
)
-- Final summary by round
SELECT 
    Round,
    COUNT(*) as total_drafted,
    SUM(CASE WHEN final_tier = 'Elite' THEN 1 ELSE 0 END) as elite_count,
    SUM(CASE WHEN final_tier = 'Hit' THEN 1 ELSE 0 END) as hit_count,
    SUM(CASE WHEN final_tier = 'Role' THEN 1 ELSE 0 END) as role_count,
    SUM(CASE WHEN final_tier = 'Bust' THEN 1 ELSE 0 END) as bust_count,
    SUM(CASE WHEN final_tier = 'Too Early' THEN 1 ELSE 0 END) as too_early_count,
    -- Calculate percentages (excluding Too Early)
    ROUND(SUM(CASE WHEN final_tier IN ('Elite', 'Hit', 'Role') THEN 1 ELSE 0 END) * 100.0 / 
          NULLIF(SUM(CASE WHEN final_tier != 'Too Early' THEN 1 ELSE 0 END), 0), 1) as success_rate,
    ROUND(SUM(CASE WHEN final_tier = 'Elite' THEN 1 ELSE 0 END) * 100.0 / 
          NULLIF(SUM(CASE WHEN final_tier != 'Too Early' THEN 1 ELSE 0 END), 0), 1) as elite_rate
FROM drafted_players
GROUP BY Round
ORDER BY Round;
"""

# Execute and display
results = pd.read_sql_query(query, conn)

print("WR HIT RATE BY DRAFT ROUND (2017-2025)")
print("=" * 100)
print(results.to_string(index=False))

# Also show overall totals
print("\n" + "=" * 100)
print("📊 OVERALL SUMMARY")

total_players = results['total_drafted'].sum()
total_evaluated = total_players - results['too_early_count'].sum()
print(f"Total WRs drafted 2017-2025: {total_players}")
print(f"Evaluated (excluding Too Early): {total_evaluated}")
print(f"Too Early (2024-2025): {results['too_early_count'].sum()}")

conn.close()

**Recalibration** <br>

This method didn't work quite right, so I decided to go a different route and measure players season tiering numerically.

In [8]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('nfl_analytics.db')

query = """
WITH player_tiers AS (
    -- Use numeric scores for proper MAX comparison
    SELECT 
        n.NAME,
        MAX(CASE 
            WHEN n.YDS >= 1200 OR n.TD >= 10 THEN 4  -- Elite
            WHEN n.YDS >= 800 OR n.TD >= 6 THEN 3    -- Hit
            WHEN n.YDS >= 400 OR n.TD >= 3 THEN 2    -- Role
            ELSE 1                                    -- Below Role
        END) as tier_score,
        MAX(n.YDS) as peak_yards,
        MAX(n.TD) as peak_tds,
        COUNT(DISTINCT n.YEAR) as seasons_in_nextgen
    FROM nextgen n
    WHERE n.POS = 'WR'
    GROUP BY n.NAME
),
drafted_players AS (
    -- Get all drafted WRs 2017-2025
    SELECT 
        d.Name as Player,
        d.Year as draft_year,
        d.Round,
        d.Pick,
        CASE 
            WHEN pt.tier_score = 4 THEN 'Elite'
            WHEN pt.tier_score = 3 THEN 'Hit'
            WHEN pt.tier_score = 2 THEN 'Role'
            WHEN pt.tier_score = 1 THEN 'Below Role'
            ELSE NULL
        END as peak_tier,
        pt.peak_yards,
        pt.peak_tds,
        pt.seasons_in_nextgen,
        -- Determine final tier with "Too Early" logic
        CASE 
            -- Too Early (2024-2025 draftees not yet proven)
            WHEN d.Year >= 2024 AND (pt.tier_score IS NULL OR pt.tier_score <= 2) THEN 'Too Early'
            -- Never appeared in NextGen = Bust
            WHEN pt.tier_score IS NULL THEN 'Bust'
            -- Appeared but never hit Role = Bust
            WHEN pt.tier_score = 1 THEN 'Bust'
            -- Everything else - map numeric back to tier
            WHEN pt.tier_score = 4 THEN 'Elite'
            WHEN pt.tier_score = 3 THEN 'Hit'
            WHEN pt.tier_score = 2 THEN 'Role'
        END as final_tier
    FROM draft_picks d
    LEFT JOIN player_tiers pt ON d.Name = pt.NAME
    WHERE d.Year BETWEEN 2017 AND 2025
        AND d.Name NOT LIKE '%pick%'
        AND d.Name NOT LIKE '%Pick%'
        AND d.Name NOT LIKE '%selection%'
)
-- Final summary by round
SELECT 
    Round,
    COUNT(*) as total_drafted,
    SUM(CASE WHEN final_tier = 'Elite' THEN 1 ELSE 0 END) as elite_count,
    SUM(CASE WHEN final_tier = 'Hit' THEN 1 ELSE 0 END) as hit_count,
    SUM(CASE WHEN final_tier = 'Role' THEN 1 ELSE 0 END) as role_count,
    SUM(CASE WHEN final_tier = 'Bust' THEN 1 ELSE 0 END) as bust_count,
    SUM(CASE WHEN final_tier = 'Too Early' THEN 1 ELSE 0 END) as too_early_count,
    -- Calculate percentages (excluding Too Early)
    ROUND(SUM(CASE WHEN final_tier IN ('Elite', 'Hit', 'Role') THEN 1 ELSE 0 END) * 100.0 / 
          NULLIF(SUM(CASE WHEN final_tier != 'Too Early' THEN 1 ELSE 0 END), 0), 1) as success_rate,
    ROUND(SUM(CASE WHEN final_tier = 'Elite' THEN 1 ELSE 0 END) * 100.0 / 
          NULLIF(SUM(CASE WHEN final_tier != 'Too Early' THEN 1 ELSE 0 END), 0), 1) as elite_rate
FROM drafted_players
GROUP BY Round
ORDER BY Round;
"""

# Execute and display
results = pd.read_sql_query(query, conn)

print("WR HIT RATE BY DRAFT ROUND (2017-2025)")
print("=" * 100)
print(results.to_string(index=False))

# Also show overall totals
print("\n" + "=" * 100)
print("OVERALL SUMMARY")

total_players = results['total_drafted'].sum()
total_evaluated = total_players - results['too_early_count'].sum()
print(f"Total WRs drafted 2017-2025: {total_players}")
print(f"Evaluated (excluding Too Early): {total_evaluated}")
print(f"Too Early (2024-2025): {results['too_early_count'].sum()}")

conn.close()

WR HIT RATE BY DRAFT ROUND (2017-2025)
 Round  total_drafted  elite_count  hit_count  role_count  bust_count  too_early_count  success_rate  elite_rate
     1             39           16         14           3           2                4          94.3        45.7
     2             46            5         16          10           9                6          77.5        12.5
     3             44            5          6           7          16               10          52.9        14.7
     4             38            1          3           5          19               10          32.1         3.6
     5             35            1          6           3          19                6          34.5         3.4
     6             47            0          5          10          22               10          40.5         0.0
     7             42            0          2           3          27               10          15.6         0.0

OVERALL SUMMARY
Total WRs drafted 2017-2025: 291
Evaluat

In order to make sure this data makes sense, I wanted to manually check myself. So I printed out all the receivers drafted in round five to make sure there was truly only one elite player drafted within the past 8 years in that round. 

In [9]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('nfl_analytics.db')

query = """
SELECT 
    Year,
    Round,
    Pick,
    Name as Player,
    Team,
    College
FROM draft_picks
WHERE Year BETWEEN 2017 AND 2025
    AND Round = 5
    AND Name NOT LIKE '%pick%'
    AND Name NOT LIKE '%Pick%'
    AND Name NOT LIKE '%selection%'
ORDER BY Year, Pick;
"""

fifth_round = pd.read_sql_query(query, conn)

print("🏈 5TH ROUND WR DRAFT PICKS (2017-2025)")
print("=" * 80)
print(fifth_round.to_string(index=False))
print(f"\n📊 Total 5th round WRs: {len(fifth_round)}")

conn.close()

🏈 5TH ROUND WR DRAFT PICKS (2017-2025)
 Year  Round  Pick                   Player       Team        College
 2017      5    22           Shelton Gibson     Eagles  West Virginia
 2017      5    27             Rodney Adams    Vikings  South Florida
 2017      5    29          Isaiah McKenzie    Broncos        Georgia
 2017      5    32          DeAngelo Yancey    Packers         Purdue
 2017      5    34             Trent Taylor      49ers Louisiana Tech
 2018      5     7            Justin Watson Buccaneers   Pennsylvania
 2018      5    22         Daurice Fountain      Colts  Northern Iowa
 2018      5    25            Jordan Lasley     Ravens           UCLA
 2018      5    37 Marquez Valdes-Scantling    Packers  South Florida
 2019      5    11           Hunter Renfrow    Raiders        Clemson
 2019      5    26                 EJ Speed      Colts Tarleton State
 2019      5    33           Darius Slayton     Giants         Auburn
 2020      5     5                 Joe Reed   Charg

I wanted to do one more check just to make sure, so I wrote a query to check the total number of "Elite" receivers to make sure we got the right number.

In [10]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('nfl_analytics.db')
query = """
    SELECT 
        n.NAME,
        MAX(CASE 
            WHEN n.YDS >= 1200 OR n.TD >= 10 THEN 4  -- Elite
            WHEN n.YDS >= 800 OR n.TD >= 6 THEN 3    -- Hit
            WHEN n.YDS >= 400 OR n.TD >= 3 THEN 2    -- Role
            ELSE 1                                    -- Below Role
        END) as tier_score,
        MAX(n.YDS) as peak_yards,
        MAX(n.TD) as peak_tds,
        COUNT(DISTINCT n.YEAR) as seasons_in_nextgen, 
        d.Year as DraftYear
    FROM nextgen n
    LEFT JOIN draft_picks d ON d.Name = n.Name
    WHERE n.POS = 'WR' AND d.Year BETWEEN 2017 AND 2025
    GROUP BY n.NAME
    ORDER BY 2 DESC, 6 DESC
"""

'''
query = """
SELECT 
    n.Name as Player,
    MAX(n.YDS) as Max_Yards,
    MAX(n.TD) as Max_TD
FROM nextgen n
LEFT JOIN draft_picks d
ON   n.Name = d.Name
WHERE d.Year BETWEEN 2017 AND 2025
AND n.POS = 'WR'
AND (n.YDS >= 1200 OR n.TD >= 10)
GROUP BY Player
ORDER BY 2 DESC, 3
"""
'''
checking = pd.read_sql_query(query, conn)

#print("YDS and TDs for all players")
#print("=" * 80)
print(checking.to_string(index=False))
#print(f"\n Total Elite WRs: {len(checking)}")

conn.close()

                    NAME  tier_score  peak_yards  peak_tds  seasons_in_nextgen  DraftYear
            Brian Thomas           4        1282        10                   2       2024
            Malik Nabers           4        1204         7                   1       2024
      Jaxon Smith-Njigba           4        1793        10                   3       2023
          Jordan Addison           4         911        10                   3       2023
              Puka Nacua           4        1715        10                   3       2023
             Zay Flowers           4        1211         5                   3       2023
            Drake London           4        1271         9                   4       2022
          George Pickens           4        1429         9                   4       2022
       Amon-Ra St. Brown           4        1515        12                   5       2021
           DeVonta Smith           4        1210         8                   5       2021
          

**Deeper Dive** <br>
Now that we know our findings match our data, we wanted to take a deeper and more analytical dive, counting the number of seasons each player was considered elite, a hit, a role player, or a bust. In this next cell, we will count the number of seasons where each player has an elite yards and/or td season, as well as hit seasons and role seasons. From there, we will determine how many seasons of elite play it will take to categorize a player as elite, etc. We are going for more consistency from players rather than just a peak season. <br>

Here is our new criteria: <br>

ELITE = 2+ seasons with elite YDS (1200+), <br>
        or 2+ seasons with elite TDs (10+), <br>
        or 1 season with elite YDS AND 1 season of elite TDs <br><br>

HIT =   2+ seasons with hit YDS (800-1199),  <br>
        or 2+ seasons with hit TDs (6-9), <br>
        or 1 season with hit YDS AND 1 season of hit TDs <br><br>

ROLE =  2+ seasons with role YDS (400-799), <br>
        or 2+ seasons with role TDs (3-5), <br>
        or 1 season with role YDS AND 1 season of role TDs <br><br>

Recent Draftees (2024-2025):

Players drafted in 2024: <br>
ELITE = 2+ Elite YDS OR tds, <br>
        or 1 Elite YDS AND 1 Elite tds<br><br>

HIT =   1 Elite YDS AND 1 Hit YDS, <br>
        1 Elite TDs AND 1 Hit TDs, <br>
        1 Elite YDS AND 1 Hit TDs,<br>
        1 Elite TDs and 1 Hit YDS,<br><br>

Too Soon = 1 Hit yds OR tds (or anything below) <br><br>

Players drafted in 2025 <br>
ELITE = Elite YDS AND TDs  <br>
HIT =   Elite YDS AND Hit TDs, <br>
        or Hit YDS AND Elite TDs <br>
Too Soon = anything else<br>


In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('nfl_analytics.db')

query = """
WITH player_tiers AS (
    -- Use numeric scores for proper MAX comparison
    SELECT 
        n.NAME,
        SUM(CASE WHEN n.YDS >= 1200 THEN 1 ELSE 0 END) as elite_yard_seasons,
        SUM(CASE WHEN n.YDS >= 800 THEN 1 ELSE 0 END) as hit_yard_seasons,
        SUM(CASE WHEN n.YDS >= 400 THEN 1 ELSE 0 END) as role_yard_seasons,
        SUM(CASE WHEN n.TD >= 10 THEN 1 ELSE 0 END) as elite_td_seasons,
        SUM(CASE WHEN n.TD >= 6 THEN 1 ELSE 0 END) as hit_td_seasons,
        SUM(CASE WHEN n.TD >= 3 THEN 1 ELSE 0 END) as role_td_seasons,
        COUNT(DISTINCT n.YEAR) as seasons_in_nextgen
    FROM nextgen n
    WHERE n.POS = 'WR'
    GROUP BY n.NAME
),
drafted_players AS (
    -- Get all drafted WRs 2017-2025
    SELECT 
        d.Name as Player,
        d.Year as draft_year,
        d.Round,
        d.Pick,    
        CASE 
            -- ELITE TIER
            WHEN pt.elite_yard_seasons >= 2 OR pt.elite_td_seasons >= 2 THEN 'Elite'
            WHEN pt.elite_yard_seasons >= 1 AND pt.elite_td_seasons >= 1 THEN 'Elite'
            -- HIT TIER
            WHEN pt.hit_yard_seasons >= 2 OR pt.hit_td_seasons >= 2 THEN 'Hit'
            WHEN pt.elite_yard_seasons >= 1 AND pt.hit_td_seasons >= 1 THEN 'Hit'
            WHEN pt.hit_yard_seasons >= 1 AND pt.elite_yard_seasons >= 1 THEN 'Hit'
            WHEN pt.hit_td_seasons >= 1 AND pt.elite_td_seasons >= 1 THEN 'Hit'
            WHEN pt.hit_yard_seasons >= 1 AND pt.elite_td_seasons >= 1 THEN 'Hit'
            WHEN pt.hit_yard_seasons >= 1 AND pt.hit_td_seasons >= 1 THEN 'Hit'
            -- ROLE TIER
            WHEN pt.role_yard_seasons >= 2 OR pt.role_td_seasons >= 2 THEN 'Role'
            WHEN pt.role_yard_seasons >= 1 AND pt.role_td_seasons >= 1 THEN 'Role'
            WHEN pt.hit_yard_seasons >= 1 AND pt.role_yard_seasons >= 1 THEN 'Role'
            WHEN pt.hit_td_seasons >= 1 AND pt.role_td_seasons >= 1 THEN 'Role'
            WHEN pt.hit_yard_seasons >= 1 AND pt.role_td_seasons >= 1 THEN 'Role'
            WHEN pt.role_yard_seasons >= 1 AND pt.hit_td_seasons >= 1 THEN 'Role'
            WHEN pt.elite_yard_seasons >= 1 AND pt.role_yard_seasons >= 1 THEN 'Role'
            WHEN pt.elite_td_seasons >= 1 AND pt.role_td_seasons >= 1 THEN 'Role'
            WHEN pt.elite_yard_seasons >= 1 AND pt.role_td_seasons >= 1 THEN 'Role'
            WHEN pt.role_yard_seasons >= 1 AND pt.elite_td_seasons >= 1 THEN 'Role'
            -- RECENT DRAFTEES
            -- 2024
            -- Elite
            WHEN d.Year = 2024 AND (pt.elite_yard_seasons >= 2 OR pt.elite_td_seasons >= 2) THEN 'Elite'
            WHEN d.Year = 2024 AND (pt.elite_yard_seasons >= 1 AND pt.elite_td_seasons >= 1) THEN 'Elite'
            -- HIT
            WHEN d.Year = 2024 AND (pt.elite_yard_seasons >= 1 AND pt.hit_yard_seasons >= 1) THEN 'Hit'
            WHEN d.Year = 2024 AND (pt.elite_td_seasons >= 1 AND pt.hit_td_seasons >= 1) THEN 'Hit'
            WHEN d.Year = 2024 AND (pt.elite_yard_seasons >= 1 AND pt.hit_td_seasons >= 1) THEN 'Hit'
            WHEN d.Year = 2024 AND (pt.hit_yard_seasons >= 1 AND pt.elite_td_seasons >= 1) THEN 'Hit'
            -- Too Soon
            WHEN d.Year = 2024 THEN 'Too Early'
            -- 2025
            -- Elite
            WHEN d.Year = 2025 AND (pt.elite_yard_seasons >= 1 AND pt.elite_td_seasons >= 1) THEN 'Elite'
            -- Hit
            WHEN d.Year = 2025 AND (pt.elite_yard_seasons >= 1 AND pt.hit_td_seasons >= 1) THEN 'Hit'
            WHEN d.Year = 2025 AND (pt.hit_yard_seasons >= 1 AND pt.elite_td_seasons >= 1) THEN 'Hit'
            -- Too Soon
            WHEN d.Year = 2025 THEN 'Too Early'
            -- BUSTS
            ELSE 'Bust'
        END as peak_tier,
        pt.seasons_in_nextgen
    FROM draft_picks d
    LEFT JOIN player_tiers pt ON d.Name = pt.NAME
    WHERE d.Year BETWEEN 2017 AND 2025
)
-- Final summary by round
SELECT 
    Round,
    COUNT(*) as total_drafted,
    SUM(CASE WHEN peak_tier = 'Elite' THEN 1 ELSE 0 END) as elite_count,
    SUM(CASE WHEN peak_tier = 'Hit' THEN 1 ELSE 0 END) as hit_count,
    SUM(CASE WHEN peak_tier = 'Role' THEN 1 ELSE 0 END) as role_count,
    SUM(CASE WHEN peak_tier = 'Bust' THEN 1 ELSE 0 END) as bust_count,
    SUM(CASE WHEN peak_tier = 'Too Early' THEN 1 ELSE 0 END) as too_early_count,
    -- Calculate percentages (excluding Too Early)
    ROUND(SUM(CASE WHEN peak_tier = 'Elite' THEN 1 ELSE 0 END) * 100.0 / 
      NULLIF(SUM(CASE WHEN peak_tier != 'Too Early' THEN 1 ELSE 0 END), 0), 1) as elite_rate,
    ROUND(SUM(CASE WHEN peak_tier = 'Hit' THEN 1 ELSE 0 END) * 100.0 / 
        NULLIF(SUM(CASE WHEN peak_tier != 'Too Early' THEN 1 ELSE 0 END), 0), 1) as hit_rate,
    ROUND(SUM(CASE WHEN peak_tier = 'Role' THEN 1 ELSE 0 END) * 100.0 / 
        NULLIF(SUM(CASE WHEN peak_tier != 'Too Early' THEN 1 ELSE 0 END), 0), 1) as role_rate,
    ROUND(SUM(CASE WHEN peak_tier = 'Bust' THEN 1 ELSE 0 END) * 100.0 / 
        NULLIF(SUM(CASE WHEN peak_tier != 'Too Early' THEN 1 ELSE 0 END), 0), 1) as bust_rate
FROM drafted_players
GROUP BY Round
ORDER BY Round;
"""

# Execute and display
results = pd.read_sql_query(query, conn)

print("WR HIT RATE BY DRAFT ROUND (2017-2025)")
print("=" * 100)
print(results.to_string(index=False))

# Also show overall totals
print("\n" + "=" * 100)
print("OVERALL SUMMARY")

total_players = results['total_drafted'].sum()
total_evaluated = total_players - results['too_early_count'].sum()
print(f"Total WRs drafted 2017-2025: {total_players}")
print(f"Evaluated (excluding Too Early): {total_evaluated}")
print(f"Too Early (2024-2025): {results['too_early_count'].sum()}")

conn.close()

From this data, I want to visually show, side by side, what the rates of elite, hit, role, and bust look like from round to round!

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Your results dataframe is already loaded from the previous cell
print("CREATING VISUALIZATION...")
print("=" * 60)

# Set up the plot
fig, ax = plt.subplots(figsize=(14, 8))

# Get the data
rounds = results['Round']
elite = results['elite_rate']
hit = results['hit_rate']
role = results['role_rate']
bust = results['bust_rate']

# Set up bar positions
x = np.arange(len(rounds))
width = 0.2

# Create bars
bars1 = ax.bar(x - width*1.5, elite, width, label='Elite', color='#FFD700', edgecolor='black')
bars2 = ax.bar(x - width/2, hit, width, label='Hit', color='#C0C0C0', edgecolor='black')
bars3 = ax.bar(x + width/2, role, width, label='Role', color='#CD7F32', edgecolor='black')
bars4 = ax.bar(x + width*1.5, bust, width, label='Bust', color='#6B7280', edgecolor='black')

# Customize the chart
ax.set_xlabel('Draft Round', fontsize=12, fontweight='bold')
ax.set_ylabel('Percentage (%)', fontsize=12, fontweight='bold')
ax.set_title('WR Draft Outcomes by Round (2017-2025)', fontsize=16, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([f'Round {r}' for r in rounds])
ax.set_ylim(0, 100)
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bars in [bars1, bars2, bars3, bars4]:
    for bar in bars:
        height = bar.get_height()
        if height > 0:  # Only label if bar has height
            ax.text(bar.get_x() + bar.get_width()/2., height + 1,
                   f'{height}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('wr_draft_hit_rates.png', dpi=300, bbox_inches='tight')
plt.show()

# Also show the raw numbers for reference
print("\n📋 RAW PERCENTAGES BY ROUND:")
print("=" * 60)
for i, row in results.iterrows():
    print(f"Round {row['Round']}: Elite={row['elite_rate']}%, Hit={row['hit_rate']}%, Role={row['role_rate']}%, Bust={row['bust_rate']}%")